In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)

In [ ]:
import os

print(os.listdir('/kaggle/input'))

In [ ]:
print(os.listdir('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce'))

# 1. IMPORT DATA

In [ ]:
import pandas as pd
orders = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv')

payments = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv')

customers = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv')

order_items = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv')

products = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv')

geolocation = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv')

sellers = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv')

reviews = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv')

category = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/product_category_name_translation.csv')

# 2: Create SQL Environment

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')

orders.to_sql('orders', conn, index=False)
payments.to_sql('payments', conn, index=False)
customers.to_sql('customers', conn, index=False)
order_items.to_sql('items', conn, index=False)
products.to_sql('products', conn, index=False)
geolocation.to_sql('geolocation', conn, index=False)
sellers.to_sql('sellers', conn, index=False)
reviews.to_sql('reviews', conn, index=False)
category.to_sql('category', conn, index=False)

In [ ]:
geolocation.columns

In [ ]:
geolocation.columns = geolocation.columns.str.strip()

In [ ]:
geolocation = geolocation.drop_duplicates(subset=['geolocation_zip_code_prefix'])

# 3: Build MASTER TABLE 

In [ ]:
query = """
SELECT 
    o.order_id,
    c.customer_unique_id,
    o.order_purchase_timestamp,
    p.payment_value,
    pr.product_category_name,
    g.geolocation_city

FROM orders o

JOIN customers c 
    ON o.customer_id = c.customer_id

JOIN payments p 
    ON o.order_id = p.order_id

JOIN items i 
    ON o.order_id = i.order_id

JOIN products pr 
    ON i.product_id = pr.product_id

JOIN geolocation g 
    ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
"""

master_df = pd.read_sql(query, conn)
master_df.head()
master_df.tail()


# STEP 4: Revenue Analysis

In [ ]:
query = """
SELECT 
    CAST(SUM(payment_value)AS INTEGER) AS total_revenue,
    ROUND(SUM(payment_value)/1000000, 2) AS revenue_million
FROM payments
"""
pd.read_sql(query, conn)

# 5: Product Analysis

In [ ]:
# Revenue by Category

In [ ]:
master_df.to_sql('master_df', conn, index=False, if_exists='replace')

In [ ]:
query = """
SELECT 
    product_category_name,
    ROUND(SUM(payment_value)/1000000, 2) AS revenue_million
FROM master_df
GROUP BY product_category_name
ORDER BY revenue_million DESC
LIMIT 10
"""
pd.read_sql(query, conn)

# Monthly Revenue trend

In [ ]:
query = """
SELECT 
    strftime('%Y-%m', order_purchase_timestamp) AS month,
    ROUND(SUM(payment_value)/1000000, 2) AS revenue_million
FROM master_df
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

# 6:Location Analysis

In [ ]:
#Revenue by City ( USING GEOLOCATION)


In [ ]:
query = """
SELECT 
    geolocation_city,
    ROUND(SUM(payment_value)/1000000, 2) AS revenue_million
FROM master_df
GROUP BY geolocation_city
ORDER BY revenue_million DESC
LIMIT 10
"""
pd.read_sql(query, conn)

# 7:Customer Behaviour(Segments)

In [ ]:
query = """
WITH customer_revenue AS (
    SELECT 
        customer_unique_id,
        SUM(payment_value) AS total_spent
    FROM master_df
    GROUP BY customer_unique_id
)

SELECT 
    CASE 
        WHEN total_spent > 1000 THEN 'High Value'
        WHEN total_spent > 500 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS customer_segment,
    COUNT(*) AS num_customers
FROM customer_revenue
GROUP BY customer_segment
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT 
    customer_unique_id,
    order_purchase_timestamp,
    LAG(order_purchase_timestamp) OVER (
        PARTITION BY customer_unique_id
        ORDER BY order_purchase_timestamp
    ) AS prev_order
FROM orders o
JOIN customers c 
ON o.customer_id = c.customer_id
"""
pd.read_sql(query, conn).head()

In [ ]:
# REPEAT VS ONE-TIME CUSTOMERS

In [ ]:
query = """
WITH customer_orders AS (
    SELECT 
        customer_unique_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM master_df
    GROUP BY customer_unique_id
)

SELECT 
    CASE 
        WHEN total_orders = 1 THEN 'One-Time Buyers'
        ELSE 'Repeat Customers'
    END AS customer_type,
    COUNT(*) AS num_customers
FROM customer_orders
GROUP BY customer_type
"""
pd.read_sql(query, conn)

In [ ]:
# DAYS BETWEEN ORDERS

In [ ]:
query = """
WITH ordered_data AS (
    SELECT 
        customer_unique_id,
        order_purchase_timestamp,
        LAG(order_purchase_timestamp) OVER (
            PARTITION BY customer_unique_id 
            ORDER BY order_purchase_timestamp
        ) AS prev_order
    FROM master_df
)

SELECT 
    customer_unique_id,
    order_purchase_timestamp,
    prev_order,
    JULIANDAY(order_purchase_timestamp) - JULIANDAY(prev_order) AS days_between_orders
FROM ordered_data
WHERE prev_order IS NOT NULL
"""
pd.read_sql(query, conn).head()

In [ ]:
# AVG TIME BETWEEN PURCHASES

In [ ]:
query = """
WITH ordered_data AS (
    SELECT 
        customer_unique_id,
        order_purchase_timestamp,
        LAG(order_purchase_timestamp) OVER (
            PARTITION BY customer_unique_id 
            ORDER BY order_purchase_timestamp
        ) AS prev_order
    FROM master_df
)

SELECT 
    ROUND(AVG(JULIANDAY(order_purchase_timestamp) - JULIANDAY(prev_order)),2) 
    AS avg_days_between_orders
FROM ordered_data
WHERE prev_order IS NOT NULL
"""
pd.read_sql(query, conn)

In [ ]:
#RETENTION SIGNAL

In [ ]:
query = """
WITH customer_orders AS (
    SELECT 
        customer_unique_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM master_df
    GROUP BY customer_unique_id
)

SELECT 
    ROUND(
        SUM(CASE WHEN total_orders > 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
    3) AS repeat_customer_percentage
FROM customer_orders
"""
pd.read_sql(query, conn)

# 8: RFM (Recency, Frequency,Monetery) Table

In [ ]:
query = """
SELECT 
    c.customer_unique_id,
    COUNT(o.order_id) AS frequency,
    SUM(p.payment_value) AS monetary,
    MAX(o.order_purchase_timestamp) AS last_order
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p ON o.order_id = p.order_id
GROUP BY c.customer_unique_id
"""
rfm_df = pd.read_sql(query, conn)
rfm_df.head()

In [ ]:
master_df.to_csv('master_dataset.csv', index=False)
rfm_df.to_csv('rfm_dataset.csv', index=False)

In [ ]:
#✅ Master dataset
#✅ Revenue insights
#✅ Product insights
#✅ Location insights
#✅ Customer behavior
#✅ RFM table